In [66]:
# Local steps to run in pycharm
# conda activate sage
# jupyter lab --no-browser --port=8888
# configure external server with the token
# sagemath (10.7)

from sage.all import *
from sage.plot.plot import graphics_array
from itertools import combinations
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'savefig.facecolor': 'white',
})

def graph_square(H):
    G = Graph()
    G.add_vertices(H.vertices())
    for u in H.vertices():
        for v in H.vertices():
            if u < v and H.distance(u,v) <= 2:
                G.add_edge(u,v)
    return G

def is_square_root(G, H):
    return graph_square(H) == G

In [67]:
def violates_partial(G, H):
    for u, v in combinations(H.vertices(), 2):
        try:
            d = H.distance(u, v)
        except (ValueError, RuntimeError):
            continue

        if H.has_edge(u, v) and not G.has_edge(u, v):
            return True
        if d <= 2 and not G.has_edge(u, v):
            return True
    return False

def find_all_square_roots(G):

    V = list(G.vertices())
    n = len(V)

    edges_G = set(tuple(sorted(e)) for e in G.edges(labels=False))

    from itertools import combinations
    all_edges = list(combinations(V,2))

    labeled_count = 0
    canonical_roots = {}

    def backtrack(idx, current_edges):

        nonlocal labeled_count

        if idx == len(all_edges):

            H = Graph()
            H.add_vertices(V)
            H.add_edges(current_edges)

            H2 = graph_square(H)
            edges_H2 = set(tuple(sorted(e)) for e in H2.edges(labels=False))

            if edges_H2 == edges_G:

                labeled_count += 1

                canon = H.canonical_label()
                key = canon.graph6_string()

                if key not in canonical_roots:
                    canonical_roots[key] = H.copy()

            return


        u,v = all_edges[idx]

        new_edges = current_edges + [(u,v)]

        H_partial = Graph()
        H_partial.add_vertices(V)
        H_partial.add_edges(new_edges)

        H2 = graph_square(H_partial)

        valid = True
        for e in H2.edges(labels=False):
            if tuple(sorted(e)) not in edges_G:
                valid = False
                break

        if valid:
            backtrack(idx+1,new_edges)

        backtrack(idx+1,current_edges)

    backtrack(0,[])

    return {
        "labeled": labeled_count,
        "non_isomorphic": list(canonical_roots.values())
    }

In [68]:
# LP with max objective function
def max_linear_program(G):
    m = MixedIntegerLinearProgram(solver='Gurobi', maximization=False)
    m.solver_parameter("Method", 2)

    x = m.new_variable(real=True, nonnegative=True)
    y = m.new_variable(real=True, nonnegative=True)
    z = m.new_variable(real=True, nonnegative=True)

    A_G = G.adjacency_matrix()
    n = A_G.nrows()
    M = n**2

    # Indicator argument
    for i in range(n):
        for j in range(n):
            if i == j:
                m.add_constraint(z[i,j] - (sum(y[i,k,j] for k in range(n)) - sum(x[i,k] for k in range(n)) + x[i,j]) == 0)
            else:
                m.add_constraint(z[i,j] - (sum(y[i,k,j] for k in range(n)) + x[i,j]) == 0)

    # Indicator function
    for i in range(n):
        for j in range(n):
            m.add_constraint(z[i,j] - M * A_G[i][j] <= 0)
            m.add_constraint(-z[i,j] + A_G[i][j] <= 0)

    # Product
    for i in range(n):
        for j in range(n):
            for k in range(n):
                m.add_constraint(y[i,k,j] - x[i,k] <= 0)
                m.add_constraint(y[i,k,j] - x[k,j] <= 0)
                m.add_constraint(- y[i,k,j] + x[i,k] + x[k,j] - 1 <= 0)

    # Symmetry
    for i in range(n):
        for j in range(n):
            m.add_constraint(x[i,j] == x[j,i])

    # Diagonal
    for i in range(n):
        m.add_constraint(x[i,i] == 0)

    # Variables
    for i in range(n):
        for j in range(n):
            m.add_constraint(x[i,j] - 1 <= 0)
            for k in range(n):
                m.add_constraint(y[i,k,j] - 1 <= 0)

    m.set_objective(- m.sum(x[i,j] for i in range(n) for j in range(n)))
    return m, x

def find_leaf_stem(G, verbose=False):
    n = G.order()
    N = {v: set(G.neighbors(v)) | {v} for v in range(n)}
    signature = {v: frozenset(N[v]) for v in range(n)}

    contained = [[False] * n for _ in range(n)]
    proper = [[False] * n for _ in range(n)]

    for u in range(n):
        for v in range(n):
            if N[u] <= N[v]:
                contained[u][v] = True
                proper[u][v] = N[u] != N[v]

    for s in range(n):
        for u in N[s] - {s}:
            if not proper[s][u]:
                continue

            for v in N[u] - {u}:
                if not contained[u][v]:
                    continue

                leaves = N[s] - {u, v}

                if all(signature[w] == signature[s] for w in leaves):
                    if verbose:
                        print(f"Leaf found: leaf={s}, stem={u}, vertex={v}, leaves={leaves}")

                    return {
                        "leaf": s,
                        "stem": u,
                        "vertex": v,
                        "leaves": leaves
                    }

    return None

# LP with unicliqual vertex fixing in maximal clique
def unicliqual_vertex_linear_program(G, verbose=False):
    m = MixedIntegerLinearProgram(solver='Gurobi', maximization=False)
    m.solver_parameter("Method", 2)

    x = m.new_variable(real=True, nonnegative=True)
    y = m.new_variable(real=True, nonnegative=True)
    z = m.new_variable(real=True, nonnegative=True)

    A_G = G.adjacency_matrix()
    n = A_G.nrows()
    M = n**2

    # Indicator argument
    for i in range(n):
        for j in range(n):
            if i == j:
                m.add_constraint(z[i,j] - (sum(y[i,k,j] for k in range(n)) - sum(x[i,k] for k in range(n)) + x[i,j]) == 0)
            else:
                m.add_constraint(z[i,j] - (sum(y[i,k,j] for k in range(n)) + x[i,j]) == 0)

    # Indicator function
    for i in range(n):
        for j in range(n):
            m.add_constraint(z[i,j] - M * A_G[i][j] <= 0)
            m.add_constraint(-z[i,j] + A_G[i][j] <= 0)

    # Product
    for i in range(n):
        for j in range(n):
            for k in range(n):
                m.add_constraint(y[i,k,j] - x[i,k] <= 0)
                m.add_constraint(y[i,k,j] - x[k,j] <= 0)
                m.add_constraint(- y[i,k,j] + x[i,k] + x[k,j] - 1 <= 0)

    # Symmetry
    for i in range(n):
        for j in range(n):
            m.add_constraint(x[i,j] == x[j,i])

    # Diagonal
    for i in range(n):
        m.add_constraint(x[i,i] == 0)

    # Variables
    for i in range(n):
        for j in range(n):
            m.add_constraint(x[i,j] - 1 <= 0)
            for k in range(n):
                m.add_constraint(y[i,k,j] - 1 <= 0)

    # Find a leaf-stem structure using the largest closed neighbourhood clique
    structure = find_leaf_stem(G, verbose=verbose)

    if structure is None:
        return m, x

    s = structure["leaf"]
    u = structure["stem"]

    m.add_constraint(x[s, u] == 1)
    m.add_constraint(x[u, s] == 1)

    if verbose:
        print(f"Fixed root edge: ({s}, {u})")

    m.set_objective(-m.sum(x[i, j] for i in range(n) for j in range(n)))

    return m, x

def vertex_deletion_linear_program(G, verbose=False):
    fixed_edges = set()

    m, x = unicliqual_vertex_linear_program(G)

    for u, v in fixed_edges:
        m.add_constraint(x[u, v] == 1)
        m.add_constraint(x[v, u] == 1)

    m.solve()

    n = G.order()
    solution_x = m.get_values(x)

    integral_edges = {
        tuple(sorted((i, j)))
        for i in range(n)
        for j in range(i + 1, n)
        if abs(solution_x[i, j] - 1) <= 1e-6
    }

    fixed_edges.update(integral_edges)

    is_integral = not any(
        abs(solution_x[i, j] - round(solution_x[i, j])) > 1e-6
        for i in range(n)
        for j in range(n)
    )

    if is_integral:
        if verbose:
            print("Original LP solution is integral.")
        return m, x

    if verbose:
        print(f"Original LP solution is fractional; fixed {len(integral_edges)} edges.")

    unfixed_degree = {
        v: sum(
            tuple(sorted((u, v))) not in fixed_edges
            for u in G.neighbors(v)
        )
        for v in G
    }

    v = min(unfixed_degree, key=lambda u: unfixed_degree[u])

    if verbose:
        print(f"Deleting v={v} with unfixed degree {unfixed_degree[v]}.")

    remaining_vertices = [u for u in G if u != v]

    G_minus = G.subgraph(remaining_vertices)
    relabel_map = {u: i for i, u in enumerate(remaining_vertices)}
    G_minus.relabel(relabel_map)

    fixed_edges_minus = {
        tuple(sorted((relabel_map[u], relabel_map[w])))
        for u, w in fixed_edges
        if u in relabel_map and w in relabel_map
    }

    if verbose:
        print("Solving LP on G-.")

    H_minus = None

    try:
        m_minus, x_minus = unicliqual_vertex_linear_program(G_minus)

        for u, w in fixed_edges_minus:
            m_minus.add_constraint(x_minus[u, w] == 1)
            m_minus.add_constraint(x_minus[w, u] == 1)

        m_minus.solve()

        n_minus = G_minus.order()
        solution_x_minus = m_minus.get_values(x_minus)

        is_integral_minus = not any(
            abs(solution_x_minus[i, j] - round(solution_x_minus[i, j])) > 1e-6
            for i in range(n_minus)
            for j in range(n_minus)
        )

        if is_integral_minus:
            H_minus = Graph(matrix([
                [round(solution_x_minus[i, j]) for j in range(n_minus)]
                for i in range(n_minus)
            ]))

            if verbose:
                print("LP(G-) is integral; using its root as first H- candidate.")
        else:
            if verbose:
                print("LP(G-) is fractional.")

    except MIPSolverException:
        if verbose:
            print("LP(G-) is infeasible.")

    roots = find_all_square_roots(G_minus).get("non_isomorphic", [])

    candidates = []

    if H_minus is not None:
        candidates.append(H_minus)

    for H in roots:
        if H_minus is None or not H.is_isomorphic(H_minus):
            candidates.append(H)

    if verbose:
        print(f"Found {len(candidates)} H- candidates.")

    for k, H_minus in enumerate(candidates):
        if verbose:
            print(f"Trying H- candidate {k + 1}/{len(candidates)}.")

        m_test, x_test = unicliqual_vertex_linear_program(G)

        for u, w in fixed_edges:
            m_test.add_constraint(x_test[u, w] == 1)
            m_test.add_constraint(x_test[w, u] == 1)

        for i, a in enumerate(remaining_vertices):
            for j, b in enumerate(remaining_vertices):
                if a != b and H_minus.has_edge(i, j):
                    m_test.add_constraint(x_test[a, b] == 1)
                    if verbose:
                        print(f"x[{a},{b}] = 1")

        try:
            m_test.solve()
        except MIPSolverException:
            if verbose:
                print("Modified LP is infeasible; trying next H-.")
            continue

        solution_x_test = m_test.get_values(x_test)

        new_fixed_edges = {
            tuple(sorted((i, j)))
            for i in range(n)
            for j in range(i + 1, n)
            if abs(solution_x_test[i, j] - 1) <= 1e-6
        }

        added_edges = new_fixed_edges - fixed_edges
        fixed_edges.update(new_fixed_edges)

        is_integral_test = not any(
            abs(solution_x_test[i, j] - round(solution_x_test[i, j])) > 1e-6
            for i in range(n)
            for j in range(n)
        )

        if is_integral_test:
            if verbose:
                print("Modified LP is integral; returning fixed LP.")
            return m_test, x_test

        if verbose:
            print(f"Modified LP is fractional; fixed {len(added_edges)} new edges.")

        if added_edges:
            m = m_test
            x = x_test
            solution_x = solution_x_test

            unfixed_degree = {
                u: sum(
                    tuple(sorted((w, u))) not in fixed_edges
                    for w in G.neighbors(u)
                )
                for u in G
            }

            v = min(unfixed_degree, key=lambda u: unfixed_degree[u])

            if verbose:
                print(f"Continuing with v={v}; unfixed degree={unfixed_degree[v]}.")

            return vertex_deletion_linear_program(G, verbose)

    if verbose:
        print("All H- candidates exhausted; returning original LP.")

    return m, x

def vertex_deletion_recursive_linear_program(G, verbose=False):
    fixed_edges = set()

    m, x = unicliqual_vertex_linear_program(G)
    m.solve()

    n = G.order()
    solution_x = m.get_values(x)

    integral_edges = {
        tuple(sorted((i, j)))
        for i in range(n)
        for j in range(i + 1, n)
        if abs(solution_x[i, j] - 1) <= 1e-6
    }

    fixed_edges.update(integral_edges)

    is_integral = not any(
        abs(solution_x[i, j] - round(solution_x[i, j])) > 1e-6
        for i in range(n)
        for j in range(n)
    )

    if is_integral:
        if verbose:
            print("Original LP solution is integral.")
        return m, x

    if verbose:
        print(f"Original LP solution is fractional; fixed {len(integral_edges)} edges.")

    unfixed_degree = {
        v: sum(
            tuple(sorted((u, v))) not in fixed_edges
            for u in G.neighbors(v)
        )
        for v in G
    }

    v = min(unfixed_degree, key=lambda u: unfixed_degree[u])

    if verbose:
        print(f"Deleting v={v} with unfixed degree {unfixed_degree[v]}.")

    remaining_vertices = [u for u in G if u != v]
    G_minus = G.subgraph(remaining_vertices)

    relabel_map = {u: i for i, u in enumerate(remaining_vertices)}
    G_minus.relabel(relabel_map)

    fixed_edges_minus = {
        tuple(sorted((relabel_map[u], relabel_map[w])))
        for u, w in fixed_edges
        if u in relabel_map and w in relabel_map
    }

    if verbose:
        print("Solving LP on G-.")

    H_minus = None

    try:
        m_minus, x_minus = unicliqual_vertex_linear_program(G_minus)

        for u, w in fixed_edges_minus:
            m_minus.add_constraint(x_minus[u, w] == 1)
            m_minus.add_constraint(x_minus[w, u] == 1)

        m_minus.solve()

        n_minus = G_minus.order()
        solution_x_minus = m_minus.get_values(x_minus)

        is_integral_minus = not any(
            abs(solution_x_minus[i, j] - round(solution_x_minus[i, j])) > 1e-6
            for i in range(n_minus)
            for j in range(n_minus)
        )

        if is_integral_minus:
            H_minus = Graph(matrix([
                [round(solution_x_minus[i, j]) for j in range(n_minus)]
                for i in range(n_minus)
            ]))

            if verbose:
                print("LP(G-) is integral; using its root as first H- candidate.")

        else:
            if verbose:
                print("LP(G-) is fractional; recursively applying vertex deletion to G-.")

            m_recursive, x_recursive = vertex_deletion_linear_program(
                G_minus,
                verbose
            )

            m_minus = m_recursive
            x_minus = x_recursive
            solution_x_minus = m_recursive.get_values(x_recursive)

            is_integral_recursive = not any(
                abs(solution_x_minus[i, j] - round(solution_x_minus[i, j])) > 1e-6
                for i in range(n_minus)
                for j in range(n_minus)
            )

            if is_integral_recursive:
                H_minus = Graph(matrix([
                    [round(solution_x_minus[i, j]) for j in range(n_minus)]
                    for i in range(n_minus)
                ]))

                if verbose:
                    print("Recursive LP(G-) is integral; using its root as first H- candidate.")
            elif verbose:
                print("Recursive LP(G-) remains fractional.")

    except MIPSolverException:
        if verbose:
            print("LP(G-) is infeasible.")

    roots = find_all_square_roots(G_minus).get("non_isomorphic", [])

    candidates = []

    if H_minus is not None:
        candidates.append(H_minus)

    for H in roots:
        if H_minus is None or not H.is_isomorphic(H_minus):
            candidates.append(H)

    if verbose:
        print(f"Found {len(candidates)} H- candidates.")

    for k, H_minus_candidate in enumerate(candidates):
        if verbose:
            print(f"Trying H- candidate {k + 1}/{len(candidates)}.")

        m_test, x_test = unicliqual_vertex_linear_program(G)

        for u, w in fixed_edges:
            m_test.add_constraint(x_test[u, w] == 1)
            m_test.add_constraint(x_test[w, u] == 1)

        for i, a in enumerate(remaining_vertices):
            for j, b in enumerate(remaining_vertices):
                if a != b and H_minus_candidate.has_edge(i, j):
                    m_test.add_constraint(x_test[a, b] == 1)

                    if verbose:
                        print(f"x[{a},{b}] = 1")

        try:
            m_test.solve()
        except MIPSolverException:
            if verbose:
                print("Modified LP is infeasible; trying next H-.")
            continue

        solution_x_test = m_test.get_values(x_test)

        new_fixed_edges = {
            tuple(sorted((i, j)))
            for i in range(n)
            for j in range(i + 1, n)
            if abs(solution_x_test[i, j] - 1) <= 1e-6
        }

        added_edges = new_fixed_edges - fixed_edges
        fixed_edges.update(new_fixed_edges)

        is_integral_test = not any(
            abs(solution_x_test[i, j] - round(solution_x_test[i, j])) > 1e-6
            for i in range(n)
            for j in range(n)
        )

        if is_integral_test:
            if verbose:
                print("Modified LP is integral; returning fixed LP.")
            return m_test, x_test

        if verbose:
            print(f"Modified LP is fractional; fixed {len(added_edges)} new edges.")

        if added_edges:
            m = m_test
            x = x_test
            solution_x = solution_x_test

    if verbose:
        print("All H- candidates exhausted; returning current LP.")

    return m, x

In [ ]:
import os
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from sage.numerical.mip import MIPSolverException
from sage.graphs.graph_generators import graphs as graph_gens
from concurrent.futures import ThreadPoolExecutor, as_completed

n = 8
ncpus = 12

dir = "vertex_deletion_enumerate_subgraphs_graph_output"
output_dir = os.path.join(dir, f"n={n}")
output_pdf = os.path.join(output_dir, f"graph_results_n={n}.pdf")
comparison_pdf = os.path.join(output_dir, f"graph_comparison_n={n}.pdf")
summary_file = os.path.join(output_dir, f"graph_summary_n={n}.txt")

os.makedirs(output_dir, exist_ok=True)

def error_plot(message):
    g = Graphics()
    g += text(message, (0, 0), fontsize=14, color="red", horizontal_alignment="center")
    g.axes(False)
    return g

def is_claw_free_graph(G):
    for v in G:
        N = G.neighbors(v)
        for a in range(len(N)):
            for b in range(a + 1, len(N)):
                for c in range(b + 1, len(N)):
                    u1, u2, u3 = N[a], N[b], N[c]
                    if not G.has_edge(u1, u2) and not G.has_edge(u1, u3) and not G.has_edge(u2, u3):
                        return False
    return True

def is_dismantlable_graph(G):
    G_1 = G.copy()
    while G_1.order() > 1:
        found = False
        for v in G_1:
            Nv = set(G_1.neighbors(v)) | {v}
            for u in G_1:
                if u != v and Nv.issubset(set(G_1.neighbors(u)) | {u}):
                    G_1.delete_vertex(v)
                    found = True
                    break
            if found:
                break
        if not found:
            return False
    return True

def process_graph(idx, G, label, known_roots):
    G = Graph(G)
    G.relabel(range(G.order()))
    nG = G.order()
    is_noninteger = False

    try:
        m, x = vertex_deletion_linear_program(G, verbose=False)
        m.solve()
        solution_x = m.get_values(x)
        A_H = matrix([[solution_x[i, j] for j in range(nG)] for i in range(nG)])
        sol_H = Graph(A_H, weighted=True)
        for u, v, w in sol_H.edges():
            sol_H.set_edge_label(u, v, round(w, 3))
        lp_status = "feasible"
        is_noninteger = any(abs(A_H[i, j] - round(A_H[i, j])) > 1e-6 for i in range(nG) for j in range(nG))
        p1 = sol_H.plot(edge_labels=True, title="Weighted LP H")
    except MIPSolverException:
        sol_H = None
        lp_status = "infeasible"
        p1 = error_plot("LP infeasible")

    num_unique_roots = len(known_roots)
    has_root = num_unique_roots > 0
    H = known_roots[0] if has_root else None

    root_properties = {
        "is_tree": None,
        "is_bipartite": None,
        "is_proper_interval": None,
        "is_girth_more_than_six": None,
        "is_block": None,
        "is_comparability": None,
        "is_dismantlable": None
    }

    if H is None:
        p2 = error_plot("No feasible H")
        edges_H = None
    else:
        p2 = H.plot(title="Feasible H")
        edges_H = H.size()
        root_properties["is_tree"] = H.is_tree()
        root_properties["is_bipartite"] = H.is_bipartite()
        root_properties["is_proper_interval"] = H.is_interval() and is_claw_free_graph(H)
        root_properties["is_girth_more_than_six"] = True if H.is_tree() else (H.girth() is not None and H.girth() > 6)
        root_properties["is_block"] = H.is_block_graph()
        root_properties["is_comparability"] = H.is_comparability()
        root_properties["is_dismantlable"] = is_dismantlable_graph(H)

    p3 = G.plot(title=f"{label} G (#{idx})")

    summary = {
        "index": idx,
        "edges_G": G.size(),
        "vertices_G": nG,
        "lp_status": lp_status,
        "has_root": has_root,
        "edges_H": edges_H,
        "is_noninteger": is_noninteger,
        "total_roots": num_unique_roots,
        "unique_roots": num_unique_roots,
        **root_properties
    }

    is_mismatch = (lp_status == "feasible" and is_noninteger) or (lp_status == "feasible" and not has_root) or (lp_status == "infeasible" and has_root)

    if is_mismatch:
        p_left = sol_H.plot(edge_labels=True, title="Weighted LP H") if sol_H is not None else error_plot("No LP solution")
        p_mid = H.plot(title="Feasible H") if H is not None else error_plot("No discrete H")
        comparison = graphics_array([p_left, p_mid, G.plot(title="Original G")])
    else:
        comparison = None

    return (p1, p2, p3), summary, comparison, is_mismatch

root_graphs = {}
all_graphs = []

for idx, G0 in enumerate(graph_gens.nauty_geng(str(n))):
    G = Graph(G0)
    G.relabel(range(G.order()))
    all_graphs.append((idx, G))

for H0 in graph_gens.nauty_geng(str(n)):
    H = Graph(H0)
    H.relabel(range(H.order()))
    G = graph_square(H)
    G.relabel(range(G.order()))
    key = G.canonical_label().graph6_string()
    if key not in root_graphs:
        root_graphs[key] = []
    if not any(H.is_isomorphic(K) for K in root_graphs[key]):
        root_graphs[key].append(H)

root_tasks = []
no_root_tasks = []

for idx, G in all_graphs:
    key = G.canonical_label().graph6_string()
    if key in root_graphs:
        root_tasks.append((idx, G, root_graphs[key]))
    else:
        no_root_tasks.append((idx, G, []))

tasks = root_tasks + no_root_tasks

print(f"Root classification: total={len(tasks)} | with roots={len(root_tasks)} | without roots={len(no_root_tasks)}")

def run_task(task):
    idx, G, known_roots = task
    return idx, process_graph(idx, G, "Original", known_roots)

results = []
mismatch_results = []
summary = {}
graph_info = {}

with ThreadPoolExecutor(max_workers=ncpus) as executor:
    futures = {executor.submit(run_task, task): task[0] for task in tasks}

    for future in as_completed(futures):
        idx, result = future.result()
        plot_G, summary_G, comp_G, mismatch_G = result

        if mismatch_G and comp_G is not None:
            mismatch_results.append((comp_G, summary_G))

        p1, p2, p3 = plot_G
        results.append((graphics_array([[p1, p2, p3]]), summary_G))

        H_exists = summary_G["has_root"]

        if summary_G["lp_status"] == "infeasible":
            LP_type = "LP infeasible"
        elif summary_G["is_noninteger"]:
            LP_type = "LP fractional"
        else:
            LP_type = "LP integer"

        summary.setdefault((H_exists, LP_type), []).append(idx)

        graph_info[idx] = {
            "H_exists": H_exists,
            "LP_type": LP_type,
            "TotalRoots": summary_G["total_roots"],
            "NonIsoRoots": summary_G["unique_roots"],
            "Tree": "Yes" if summary_G["is_tree"] else "No",
            "Bip": "Yes" if summary_G["is_bipartite"] else "No",
            "PropInt": "Yes" if summary_G["is_proper_interval"] else "No",
            "Girth>6": "Yes" if summary_G["is_girth_more_than_six"] else "No",
            "Block": "Yes" if summary_G["is_block"] else "No",
            "Comparability": "Yes" if summary_G["is_comparability"] else "No",
            "Dismantlable": "Yes" if summary_G["is_dismantlable"] else "No"
        }

        print(f"Graph {summary_G['index']:>4} | LP={summary_G['lp_status']:<10} | Root={'Yes' if summary_G['has_root'] else 'No':<3} | NonInt={'Yes' if summary_G['is_noninteger'] else 'No':<3} | {'MISMATCH' if mismatch_G else 'OK'}")


if n < 8:
    with open(summary_file, "w") as f:
        header = f"{'Graph':>5} | {'Verts':>5} | {'Edges(G)':>8} | {'Roots':>7} | {'NonIso':>7} | {'Edges(H)':>8} | {'LP':>12} | {'Root':>5} | {'NonInt':>7} | {'Tree':>5} | {'Bip':>5} | {'PropInt':>7} | {'Girth>6':>8} | {'Block':>5} | {'Comparability':>13} | {'Dismantlable':>12}\n"
        f.write(header)
        f.write("-" * len(header) + "\n")
        for _, s in sorted(results, key=lambda z: z[1]["index"]):
            f.write(f"{s['index']:>5} | {s['vertices_G']:>5} | {s['edges_G']:>8} | {s['total_roots']:>7} | {s['unique_roots']:>7} | {str(s['edges_H']) if s['edges_H'] is not None else 'NA':>8} | {s['lp_status']:<12} | {'Yes' if s['has_root'] else 'No':>5} | {'Yes' if s['is_noninteger'] else 'No':>7} | {'Yes' if s['is_tree'] else 'No':>5} | {'Yes' if s['is_bipartite'] else 'No':>5} | {'Yes' if s['is_proper_interval'] else 'No':>7} | {'Yes' if s['is_girth_more_than_six'] else 'No':>8} | {'Yes' if s['is_block'] else 'No':>5} | {'Yes' if s['is_comparability'] else 'No':>13} | {'Yes' if s['is_dismantlable'] else 'No':>12}\n")
            f.write("-" * len(header) + "\n")
    print(f"Summary written to {summary_file}")

if n < 8:
    analysis_file = os.path.join(output_dir, f"graph_analysis_n={n}.txt")

    with open(analysis_file, "w", encoding="utf-8") as f:
        f.write("==============================\n")
        f.write("Root Classification\n")
        f.write("==============================\n")

        for (H_exists, LP_type), graphs in sorted(summary.items()):
            f.write(f"H exists: {str(H_exists):<5} | {LP_type:<15} → {len(graphs)} graphs\n")

        properties = [
            ("Tree", "Tree"),
            ("Bip", "Bipartite"),
            ("PropInt", "Proper Interval"),
            ("Girth>6", "Girth > 6"),
            ("Block", "Block Graph"),
            ("Comparability", "Comparability"),
            ("Dismantlable", "Dismantlable")
        ]

        f.write("\n==============================\n")
        f.write("Fractional LP Analysis\n")
        f.write("==============================\n")

        for key, label in properties:
            H_true_yes = H_true_no = H_false_yes = H_false_no = 0

            for info in graph_info.values():
                if info["LP_type"] != "LP fractional":
                    continue

                if info["H_exists"]:
                    if info[key] == "Yes":
                        H_true_yes += 1
                    else:
                        H_true_no += 1
                else:
                    if info[key] == "Yes":
                        H_false_yes += 1
                    else:
                        H_false_no += 1

            f.write(f"\n{label} for fractional solutions\n")
            f.write(f"H=True  | {label}=Yes → {H_true_yes}\n")
            f.write(f"H=True  | {label}=No  → {H_true_no}\n")
            f.write(f"H=False | {label}=Yes → {H_false_yes}\n")
            f.write(f"H=False | {label}=No  → {H_false_no}\n")

        f.write("\n==============================\n")
        f.write("Fractional Solutions with Roots\n")
        f.write("==============================\n")

        for idx in sorted(graph_info):
            info = graph_info[idx]

            if info["LP_type"] != "LP fractional" or not info["H_exists"]:
                continue

            warning = (
                info["Tree"] == "No"
                and info["Bip"] == "No"
                and info["PropInt"] == "No"
                and info["Girth>6"] == "No"
                and info["Block"] == "No"
            )

            warn_symbol = " ⚠" if warning else ""

            f.write(f"\nGraph {idx}{warn_symbol}\n")
            f.write(f"  Original → H exists: {info['H_exists']} | {info['LP_type']}\n")
            f.write(f"  Roots: {info['TotalRoots']}\n")
            f.write(
                f"  Properties: Tree={info['Tree']} "
                f"Bip={info['Bip']} "
                f"PI={info['PropInt']} "
                f"Girth>6={info['Girth>6']} "
                f"Block={info['Block']} "
                f"Comparability={info['Comparability']} "
                f"Dismantlable={info['Dismantlable']}\n"
            )

    print(f"Analysis written to {analysis_file}")

    with PdfPages(output_pdf) as pdf:
        for p, sG in results:
            fig = p.matplotlib(figsize=(12, 4))
            pdf.savefig(fig)
            plt.close(fig)

    print(f"Combined {len(results)} graphs into {output_pdf}")

    if mismatch_results:
        with PdfPages(comparison_pdf) as pdf:
            for comp, s in mismatch_results:
                fig = comp.matplotlib(figsize=(12, 4))
                plt.suptitle(
                    f"Graph {s['index']} | "
                    f"LP={s['lp_status']} | "
                    f"Root={s['has_root']} | "
                    f"{'Non-integer LP' if s['is_noninteger'] else ''}"
                )
                pdf.savefig(fig)
                plt.close(fig)

        print(f"Comparison mismatches saved to {comparison_pdf}")

else:
    print("==============================")
    print("Root Classification")
    print("==============================")

    for (H_exists, LP_type), graphs in sorted(summary.items()):
        print(f"H exists: {str(H_exists):<5} | {LP_type:<15} → {len(graphs)} graphs")

    properties = [
        ("Tree", "Tree"),
        ("Bip", "Bipartite"),
        ("PropInt", "Proper Interval"),
        ("Girth>6", "Girth > 6"),
        ("Block", "Block Graph"),
        ("Comparability", "Comparability"),
        ("Dismantlable", "Dismantlable")
    ]

    print("\n==============================")
    print("Fractional LP Analysis")
    print("==============================")

    for key, label in properties:
        H_true_yes = H_true_no = H_false_yes = H_false_no = 0

        for info in graph_info.values():
            if info["LP_type"] != "LP fractional":
                continue

            if info["H_exists"]:
                if info[key] == "Yes":
                    H_true_yes += 1
                else:
                    H_true_no += 1
            else:
                if info[key] == "Yes":
                    H_false_yes += 1
                else:
                    H_false_no += 1

        print(f"\n{label} for fractional solutions")
        print(f"H=True  | {label}=Yes → {H_true_yes}")
        print(f"H=True  | {label}=No  → {H_true_no}")
        print(f"H=False | {label}=Yes → {H_false_yes}")
        print(f"H=False | {label}=No  → {H_false_no}")

    print("\n==============================")
    print("Fractional Solutions with Roots")
    print("==============================")

    for idx in sorted(graph_info):
        info = graph_info[idx]

        if info["LP_type"] != "LP fractional" or not info["H_exists"]:
            continue

        warning = (
            info["Tree"] == "No"
            and info["Bip"] == "No"
            and info["PropInt"] == "No"
            and info["Girth>6"] == "No"
            and info["Block"] == "No"
        )

        warn_symbol = " ⚠" if warning else ""

        print(f"\nGraph {idx}{warn_symbol}")
        print(f"  Original → H exists: {info['H_exists']} | {info['LP_type']}")
        print(f"  Roots: {info['TotalRoots']}")
        print(
            f"  Properties: Tree={info['Tree']} "
            f"Bip={info['Bip']} "
            f"PI={info['PropInt']} "
            f"Girth>6={info['Girth>6']} "
            f"Block={info['Block']} "
            f"Comparability={info['Comparability']} "
            f"Dismantlable={info['Dismantlable']}"
        )

Root classification: total=12346 | with roots=307 | without roots=12039
Graph 1122 | LP=feasible   | Root=Yes | NonInt=No  | OK
Graph 2252 | LP=feasible   | Root=Yes | NonInt=No  | OK
Graph  958 | LP=feasible   | Root=Yes | NonInt=No  | OK
Graph 2149 | LP=feasible   | Root=Yes | NonInt=No  | OK
Graph  909 | LP=feasible   | Root=Yes | NonInt=No  | OK
Graph   29 | LP=feasible   | Root=Yes | NonInt=No  | OK
Graph 1550 | LP=feasible   | Root=Yes | NonInt=No  | OK
Graph    1 | LP=feasible   | Root=Yes | NonInt=No  | OK
Graph 1134 | LP=feasible   | Root=Yes | NonInt=No  | OK
Graph 1005 | LP=feasible   | Root=Yes | NonInt=No  | OK
Graph  266 | LP=feasible   | Root=Yes | NonInt=No  | OK
Graph   87 | LP=feasible   | Root=Yes | NonInt=No  | OK
Graph    8 | LP=feasible   | Root=Yes | NonInt=No  | OK
Graph 1245 | LP=feasible   | Root=Yes | NonInt=No  | OK
Graph  183 | LP=feasible   | Root=Yes | NonInt=No  | OK
Graph  912 | LP=feasible   | Root=Yes | NonInt=No  | OK
Graph    0 | LP=feasible   | Roo

In [ ]:
n = 8
filename = f"vertex_deletion_enumerate_subgraphs_graph_output/n={n}/graph_summary_n={n}.txt"

summary = {}
graph_info = {}
iso_effects = []
original_status = {}
current_graph = None
total_graphs = 0

with open(filename, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()

        if not line or line.startswith("-") or line.startswith("SUMMARY") or line.startswith("Graph"):
            continue

        parts = [p.strip() for p in line.split("|")]

        if len(parts) != 16:
            continue

        (
            graph,
            verts,
            edges_g,
            roots_total,
            roots_unique,
            edges_h,
            lp,
            root,
            nonint,
            tree,
            bip,
            propint,
            girth6,
            block,
            comparability,
            dismantlable
        ) = parts

        idx = int(graph)
        current_graph = idx

        total_graphs += 1

        roots_total = int(roots_total)
        roots_unique = int(roots_unique)

        H_exists = (root == "Yes")

        if lp == "infeasible":
            LP_type = "LP infeasible"
        elif nonint == "Yes":
            LP_type = "LP fractional"
        else:
            LP_type = "LP integer"

        key = (H_exists, LP_type)
        summary.setdefault(key, []).append(idx)

        graph_info[idx] = {
            "H_exists": H_exists,
            "LP_type": LP_type,
            "TotalRoots": roots_total,
            "NonIsoRoots": roots_unique,
            "Tree": tree,
            "Bip": bip,
            "PropInt": propint,
            "Girth>6": girth6,
            "Block": block,
            "Comparability": comparability,
            "Dismantlable": dismantlable
        }

for (H_exists, LP_type), graphs in sorted(summary.items()):
    print(
        f"H exists: {str(H_exists):<5} | "
        f"{LP_type:<15} → "
        f"{len(graphs)} graphs"
    )

properties = [
    ("Tree", "Tree"),
    ("Bip", "Bipartite"),
    ("PropInt", "Proper Interval"),
    ("Girth>6", "Girth > 6"),
    ("Block", "Block Graph"),
    ("Comparability", "Comparability"),
    ("Dismantlable", "Dismantlable")
]

for key, label in properties:

    H_true_yes = 0
    H_true_no = 0
    H_false_yes = 0
    H_false_no = 0

    for info in graph_info.values():

        if info["LP_type"] != "LP fractional":
            continue

        H = info["H_exists"]
        val = info[key]

        if H:
            if val == "Yes":
                H_true_yes += 1
            else:
                H_true_no += 1
        else:
            if val == "Yes":
                H_false_yes += 1
            else:
                H_false_no += 1

    print(f"\n{label} for fractional solutions")
    print(f"H=True  | {label}=Yes → {H_true_yes}")
    print(f"H=True  | {label}=No  → {H_true_no}")
    print(f"H=False | {label}=Yes → {H_false_yes}")
    print(f"H=False | {label}=No  → {H_false_no}")

print()

for idx in sorted(graph_info):

    info = graph_info[idx]

    if not (not info["LP_type"] == "LP integer" and info["H_exists"]):
        continue

    warning = (
        info["Tree"] == "No"
        and info["Bip"] == "No"
        and info["PropInt"] == "No"
        and info["Girth>6"] == "No"
        and info["Block"] == "No"
    )

    warn_symbol = " ⚠" if warning else ""

    print(f"Graph {idx}{warn_symbol}")

    print(
        f"  Original → "
        f"H exists: {info['H_exists']} | "
        f"{info['LP_type']}"
    )

    print(
        f"  Roots: {info['TotalRoots']} "
    )

    print(
        f"  Properties: "
        f"Tree={info['Tree']} "
        f"Bip={info['Bip']} "
        f"PI={info['PropInt']} "
        f"Girth>6={info['Girth>6']} "
        f"Block={info['Block']} "
        f"Comparability={info['Comparability']} "
        f"Dismantlable={info['Dismantlable']}"
    )

    print()

In [ ]:
%matplotlib inline

def triangles_per_vertex(G):

    counts = {v: 0 for v in G.vertices()}

    for i, j, k in combinations(G.vertices(), 3):

        if G.has_edge(i,j) and G.has_edge(i,k) and G.has_edge(j,k):

            counts[i] += 1
            counts[j] += 1
            counts[k] += 1

    return counts

def display_graph(idx=None, n=None, G=None):

    if G is None:
        if idx is None or n is None:
            raise ValueError("Provide either G or (idx, n).")

        for i, g in enumerate(graph_gens.nauty_geng(str(n))):
            if i == idx:
                G = Graph(g)
                break

        if G is None:
            print("Graph not found.")
            return
    else:
        G = Graph(G)

    G.relabel(range(G.order()))

    print("==============================")
    print(f"Graph {idx}")
    print("==============================")
    print("Vertices:", G.order())
    print("Edges:", G.size())

    H_lp = None
    lp_type = "unknown"

    try:

        m, x = vertex_deletion_recursive_linear_program(G, verbose=True)

        m.solve()

        sol = m.get_values(x)

        A = matrix([[sol[i, j] for j in range(G.order())] for i in range(G.order())])

        fractional = any(
            abs(A[i, j] - round(A[i, j])) > 1e-6
            for i in range(G.order())
            for j in range(G.order())
        )

        lp_type = "fractional" if fractional else "integer"

        H_lp = Graph(A, weighted=True)

        for (u, v, w) in H_lp.edges():
            H_lp.set_edge_label(u, v, round(w, 3))

        print("LP status: feasible")
        print("LP type:", lp_type)

    except MIPSolverException:

        print("LP status: infeasible")
        lp_type = "infeasible"

    root_info = find_all_square_roots(G)

    total_roots = root_info["labeled"]
    roots = root_info["non_isomorphic"]

    print("\nSquare roots")
    print("Total labeled:", total_roots)
    print("Non-isomorphic:", len(roots))

    print("\nTriangles per vertex in square graph G")
    tri_G = triangles_per_vertex(G)
    for v in sorted(tri_G):
        print(f"v{v}: {tri_G[v]}")

    print("\nShowing original graph G")

    p = G.plot(
        vertex_size=120,
        edge_thickness=2,
        vertex_labels=True,
        figsize=[8,8],
        title=f"G (Graph #{idx})"
    )
    display(p.matplotlib())

    if H_lp is not None:
        print("\nShowing LP solution graph H_lp")

        p = H_lp.plot(
            vertex_size=120,
            edge_thickness=2,
            edge_labels=True,
            vertex_labels=True,
            figsize=[8,8],
            title=f"H_lp ({lp_type})"
        )
        display(p.matplotlib())

    for i, H in enumerate(roots):

        print(f"\nShowing root {i+1}")

        is_tree = H.is_tree()
        is_bip = H.is_bipartite()

        is_interval = H.is_interval()
        is_claw_free = is_claw_free_graph(H)
        is_prop_int = is_interval and is_claw_free

        if is_tree:
            girth_gt6 = True
        else:
            g = H.girth()
            girth_gt6 = (g is not None and g > 6)

        is_block = H.is_block_graph()
        is_comp = H.is_comparability()
        is_dism = is_dismantlable_graph(H)

        print("Properties:")
        print(f"  Tree:            {is_tree}")
        print(f"  Bipartite:       {is_bip}")
        print(f"  Proper Interval: {is_prop_int}")
        print(f"  Girth > 6:       {girth_gt6}")
        print(f"  Block Graph:     {is_block}")
        print(f"  Comparability:   {is_comp}")
        print(f"  Dismantlable:    {is_dism}")

        p = H.plot(
            vertex_size=120,
            edge_thickness=2,
            vertex_labels=True,
            figsize=[8,8],
            title=f"Square root H{i+1}"
        )
        display(p.matplotlib())

display_graph(10, 6)
# H = Graph()
# H.add_edges([
#     (0,1),(0,2),(0,3),
#     (1,2),(1,3),
#     (2,3)
# ])
# H.add_edges([
#     (0,4),
#     (1,5),
#     (2,6),
#     (3,7)
# ])
# G = graph_square(H)
# display_graph(G=G)